In [1]:
import torch
print(torch.__version__)

2.8.0+cu129


In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch_geometric.data import HeteroData
import pickle
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.utils import negative_sampling
from torch_geometric.transforms import AddSelfLoops, ToUndirected
with open('bio_graph_raw.pkl', 'rb') as f:
    G = pickle.load(f)

C:\Users\tirth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
chemicals = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'chemical'])
diseases = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'disease'])
genes = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'gene'])

chem_idx = {c: i for i, c in enumerate(chemicals)}
disease_idx = {d: i for i, d in enumerate(diseases)}
gene_idx = {g: i for i, g in enumerate(genes)}

print(chem_idx)
print(disease_idx)
print(gene_idx)

# Extract node embeddings
chem_features = np.array([G.nodes[c]['x'] for c in chemicals], dtype=np.float32)
disease_features = np.array([G.nodes[d]['x'] for d in diseases], dtype=np.float32)
gene_features = np.array([G.nodes[g]['x'] for g in genes], dtype=np.float32)

# Extract edges
chem_gene_edges = []
chem_disease_edges = []
gene_disease_edges = []
for u, v, data in G.edges(data=True):
    edge_type = data['edge_type']
    if edge_type == 'chem_gene':
        chem_gene_edges.append([chem_idx[u], gene_idx[v]])
    elif edge_type == 'chem_disease':
        chem_disease_edges.append([chem_idx[u], disease_idx[v]])
    elif edge_type == 'gene_disease':
        gene_disease_edges.append([gene_idx[u], disease_idx[v]])


# Convert to tensors
chem_gene_edges = torch.tensor(chem_gene_edges, dtype=torch.long).t()
chem_disease_edges = torch.tensor(chem_disease_edges, dtype=torch.long).t()
gene_disease_edges = torch.tensor(gene_disease_edges, dtype=torch.long).t()


{'C000015': 0, 'C000025': 1, 'C000050': 2, 'C000061': 3, 'C000081': 4, 'C000090': 5, 'C000109': 6, 'C000121': 7, 'C000123': 8, 'C000152': 9, 'C000154': 10, 'C000188': 11, 'C000228': 12, 'C000297': 13, 'C000298': 14, 'C000380': 15, 'C000433': 16, 'C000449': 17, 'C000470': 18, 'C000474': 19, 'C000475': 20, 'C000481': 21, 'C000482': 22, 'C000483': 23, 'C000488': 24, 'C000490': 25, 'C000499': 26, 'C000505': 27, 'C000511': 28, 'C000515': 29, 'C000516': 30, 'C000518': 31, 'C000520': 32, 'C000536': 33, 'C000541': 34, 'C000543': 35, 'C000588486': 36, 'C000588503': 37, 'C000588556': 38, 'C000588664': 39, 'C000588695': 40, 'C000588741': 41, 'C000588809': 42, 'C000588918': 43, 'C000589002': 44, 'C000589029': 45, 'C000589147': 46, 'C000589162': 47, 'C000589253': 48, 'C000589451': 49, 'C000589472': 50, 'C000589490': 51, 'C000589596': 52, 'C000589651': 53, 'C000589733': 54, 'C000589878': 55, 'C000589977': 56, 'C000590225': 57, 'C000590451': 58, 'C000590645': 59, 'C000590659': 60, 'C000590786': 61, '

In [4]:
print(chem_features.shape)
print(disease_features.shape)
print(gene_features.shape)

# max_dim = max(chem_features.shape[1], disease_features.shape[1], gene_features.shape[1])
# chem_features = np.pad(chem_features, ((0, 0), (0, max_dim - chem_features.shape[1])), mode='constant')
# disease_features = np.pad(disease_features, ((0, 0), (0, max_dim - disease_features.shape[1])), mode='constant')
# gene_features = np.pad(gene_features, ((0, 0), (0, max_dim - gene_features.shape[1])), mode='constant')


# print(chem_features.shape)
# print(disease_features.shape)
# print(gene_features.shape)

(13943, 3591)
(7239, 2605)
(28151, 2635)


In [ ]:
data = HeteroData()
data['chemical'].x = torch.tensor(chem_features, dtype=torch.float)
data['disease'].x = torch.tensor(disease_features, dtype=torch.float)
data['gene'].x = torch.tensor(gene_features, dtype=torch.float)
data['chemical', 'chem_gene', 'gene'].edge_index = chem_gene_edges
data['chemical', 'chem_disease', 'disease'].edge_index = chem_disease_edges
data['gene', 'gene_disease', 'disease'].edge_index = gene_disease_edges
print(data)

data = ToUndirected()(data)
data = AddSelfLoops()(data)
print(data)

In [ ]:
if isinstance(data, HeteroData):
    for edge_type in data.edge_types:
        if 'edge_index' in data[edge_type]:
            data[edge_type].edge_index = data[edge_type].edge_index.contiguous()

# If 'data' is a homogeneous Data object (for completeness, though your code suggests hetero)
else:
    data.edge_index = data.edge_index.contiguous()

In [ ]:
from torch_geometric.transforms import RandomLinkSplit
target_edge = ('chemical', 'chem_disease', 'disease')
rev_edge = ('disease', 'rev_chem_disease', 'chemical')

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    edge_types=[target_edge],
    rev_edge_types=[rev_edge]
)
train_data, val_data, test_data = transform(data)

In [ ]:
class HeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, metadata):
        super().__init__()
        # Define HeteroConv for each edge type
        self.conv1 = HeteroConv({
            ('chemical', 'chem_gene', 'gene'): SAGEConv((-1, -1), hidden_channels),
            ('chemical', 'chem_disease', 'disease'): SAGEConv((-1, -1), hidden_channels),
            ('gene', 'gene_disease', 'disease'): SAGEConv((-1, -1), hidden_channels),
            ('gene', 'rev_chem_gene', 'chemical'): SAGEConv((-1, -1), hidden_channels),
            ('disease', 'rev_chem_disease', 'chemical'): SAGEConv((-1, -1), hidden_channels),
            ('disease', 'rev_gene_disease', 'gene'): SAGEConv((-1, -1), hidden_channels)
        }, aggr='sum')
        self.conv2 = HeteroConv({
            ('chemical', 'chem_gene', 'gene'): SAGEConv(hidden_channels, hidden_channels),
            ('chemical', 'chem_disease', 'disease'): SAGEConv(hidden_channels, hidden_channels),
            ('gene', 'gene_disease', 'disease'): SAGEConv(hidden_channels, hidden_channels),
            ('gene', 'rev_chem_gene', 'chemical'): SAGEConv(hidden_channels, hidden_channels),
            ('disease', 'rev_chem_disease', 'chemical'): SAGEConv(hidden_channels, hidden_channels),
            ('disease', 'rev_gene_disease', 'gene'): SAGEConv(hidden_channels, hidden_channels)
        }, aggr='sum')
    
    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: x.relu() for key, x in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict


In [ ]:
class HeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, metadata):
        super().__init__()

        self.proj = torch.nn.ModuleDict({
            'chemical': torch.nn.Linear(data['chemical'].x.size(1), hidden_channels),
            'disease': torch.nn.Linear(data['disease'].x.size(1), hidden_channels),
            'gene': torch.nn.Linear(data['gene'].x.size(1), hidden_channels),
        })

        self.conv1 = HeteroConv({
            edge_type: SAGEConv(hidden_channels, hidden_channels)
            for edge_type in metadata[1]
        }, aggr='sum')

        self.conv2 = HeteroConv({
            edge_type: SAGEConv(hidden_channels, hidden_channels)
            for edge_type in metadata[1]
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        # Project to shared latent space
        x_dict = {k: self.proj[k](v) for k, v in x_dict.items()}

        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: v.relu() for k, v in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict


In [ ]:
from torch_geometric.loader import LinkNeighborLoader

def predict_edges(drug_emb, disease_emb, edge_index):
    row, col = edge_index
    return (drug_emb[row] * disease_emb[col]).sum(dim=-1)

def train(model, data, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = 0
    
    # Use LinkNeighborLoader for chemical-disease edges
    # loader = LinkNeighborLoader(
    #     data,
    #     num_neighbors=[30, 10],
    #     batch_size=128,
    #     edge_label_index=(('chemical', 'chem_disease', 'disease'), data['chemical', 'chem_disease', 'disease'].edge_index),
    #     shuffle=True
    # )
    
    train_loader = LinkNeighborLoader(
        data=data,
        num_neighbors=[6000,20],
        neg_sampling_ratio=1.0,
        edge_label_index=(target_edge, data[target_edge].edge_label_index), 
        edge_label=data[target_edge].edge_label,
        batch_size=64,
        shuffle=True,
    )
    for batch in tqdm(train_loader):
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        out = model(batch.x_dict, batch.edge_index_dict)
        drug_emb = out['chemical']
        disease_emb = out['disease']
        
        # Positive edges
        pos_edge_index = batch['chemical', 'chem_disease', 'disease'].edge_index
        if pos_edge_index.size(1) == 0:
            continue
        pos_score = predict_edges(drug_emb, disease_emb, pos_edge_index)
        
        # Negative edges
        neg_edge_index = negative_sampling(
            edge_index=pos_edge_index,
            num_nodes=(batch['chemical'].num_nodes, batch['disease'].num_nodes),
            num_neg_samples=pos_edge_index.size(1)
        )
        neg_score = predict_edges(drug_emb, disease_emb, neg_edge_index)
        
        # Loss
        scores = torch.cat([pos_score, neg_score])
        labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).to(device)
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else float('inf')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HeteroGNN(hidden_channels=64, metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
data = data.to(device)

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
num_epochs = 2


for epoch in range(num_epochs):
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    loss = train(model, train_data, optimizer, device)
    print(f'Epoch {epoch+1:03d}, Loss: {loss:.4f}')
    
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch+1:03d}.pt')

print("Training complete! Models saved in 'checkpoints' directory.")

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader  # Changed to NeighborLoader
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score
from scipy.special import expit  # Stable sigmoid
import numpy as np
import os

def load_model(model, optimizer, checkpoint_path, device):
    """Load model and optimizer state from a checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    print(f"Loaded checkpoint from {checkpoint_path}, epoch {epoch}, loss: {loss:.4f}")
    return model, optimizer

import random
def evaluate(model, data, device, batch_size=128, sample_size=10000):
    model.eval()
    tp, fp, tn, fn = 0, 0, 0, 0
    pos_samples = []
    neg_samples = []
    pos_count = 0
    neg_count = 0
    
    loader = LinkNeighborLoader(
        data,
        num_neighbors=[6000,20],  # Your increased value
        batch_size=batch_size,
        edge_label_index=(('chemical', 'chem_disease', 'disease'), data['chemical', 'chem_disease', 'disease'].edge_index),
        shuffle=False
    )
    
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict)
            drug_emb = out['chemical']
            disease_emb = out['disease']
            
            pos_edge_index = batch['chemical', 'chem_disease', 'disease'].edge_index
            if pos_edge_index.size(1) == 0:
                continue
            pos_scores = predict_edges(drug_emb, disease_emb, pos_edge_index)
            pos_probs = torch.sigmoid(pos_scores)  # Stable in Torch
            pos_preds = (pos_probs > 0.5).long()
            tp += (pos_preds == 1).sum().item()
            fn += (pos_preds == 0).sum().item()
            
            neg_edge_index = negative_sampling(
                edge_index=pos_edge_index,
                num_nodes=(batch['chemical'].num_nodes, batch['disease'].num_nodes),
                num_neg_samples=pos_edge_index.size(1)
            )
            neg_scores = predict_edges(drug_emb, disease_emb, neg_edge_index)
            neg_probs = torch.sigmoid(neg_scores)
            neg_preds = (neg_probs > 0.5).long()
            fp += (neg_preds == 1).sum().item()
            tn += (neg_preds == 0).sum().item()
            
            # Reservoir sampling for AUC approximation
            for score in pos_scores.cpu().tolist():
                pos_count += 1
                if len(pos_samples) < sample_size:
                    pos_samples.append(score)
                else:
                    j = random.randint(0, pos_count - 1)
                    if j < sample_size:
                        pos_samples[j] = score
            
            for score in neg_scores.cpu().tolist():
                neg_count += 1
                if len(neg_samples) < sample_size:
                    neg_samples.append(score)
                else:
                    j = random.randint(0, neg_count - 1)
                    if j < sample_size:
                        neg_samples[j] = score
    
    # Compute exact non-AUC metrics from counts
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    balanced_acc = (recall + (tn / (tn + fp) if (tn + fp) > 0 else 0)) / 2
    
    # Approximate ROC-AUC from sampled scores
    if len(pos_samples) > 0 and len(neg_samples) > 0:
        num_pairs = len(pos_samples) * len(neg_samples)
        greater = sum(1 for p in pos_samples for n in neg_samples if p > n)
        equal = sum(1 for p in pos_samples for n in neg_samples if p == n)
        roc_auc = (greater + 0.5 * equal) / num_pairs
    else:
        roc_auc = 0.5  # Default if no samples
    
    return {
        'roc_auc': roc_auc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'balanced_accuracy': balanced_acc
    }


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = HeteroGNN(hidden_channels=64, metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

checkpoint_path = './checkpoints/model_epoch_001.pt'
model, optimizer = load_model(model, optimizer, checkpoint_path, device)

metrics = evaluate(model, test_data, device, batch_size=128)
print("Evaluation Metrics:")
print(f"ROC-AUC (approx): {metrics['roc_auc']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")
print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")

In [ ]:
def predict_top_drugs(model, data, disease_id, chem_idx, disease_idx, top_k=5, device='cuda', batch_size=1, num_neighbors=[6000 , 20]):
    model.eval()
    if disease_id not in disease_idx:
        raise ValueError(f"Disease ID {disease_id} not found in disease_idx.")
    
    disease_global_idx = torch.tensor([disease_idx[disease_id]], dtype=torch.long)
    
    # Sample subgraph around the disease (ensures disease is included)
    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=('disease', disease_global_idx),
        shuffle=False
    )
    
    all_scores = []
    all_chem_indices = []
    
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict)
            
            if 'chemical' not in out or out['chemical'].size(0) == 0:
                continue  # Skip if no chemicals sampled
            
            # Disease is always the input, so local idx 0
            disease_emb = out['disease'][0].unsqueeze(0)
            drug_emb = out['chemical']
            scores = (drug_emb * disease_emb).sum(dim=1).cpu().numpy()
            chem_globals = batch['chemical'].n_id.cpu().numpy()
            all_scores.extend(scores)
            all_chem_indices.extend(chem_globals)
    
    if len(all_scores) == 0:
        raise ValueError(f"No chemicals sampled for disease {disease_id}. Increase num_neighbors.")
    
    # Get top-k among sampled
    top_k_indices = np.argsort(all_scores)[::-1][:top_k+1]
    top_k_scores = [all_scores[i] for i in top_k_indices]
    top_k_chem_globals = [all_chem_indices[i] for i in top_k_indices]
    
    idx_to_chem = {i: c for c, i in chem_idx.items()}
    top_k_drugs = [(idx_to_chem.get(idx, 'Unknown'), score) for idx, score in zip(top_k_chem_globals, top_k_scores)]
    
    return top_k_drugs

In [ ]:
disease_id = diseases[0]  # Replace with your ID
top_k = 10
top_drugs = predict_top_drugs(model, data, disease_id, chem_idx, disease_idx, top_k=top_k, device=device, num_neighbors=[6000, 30])
print(f"\nTop {top_k} drugs (among sampled) for disease {disease_id}:")
for drug_id, score in top_drugs:
    prob = expit(score)
    print(f"Drug: {drug_id}, Score: {score:.4f} (Prob: {prob:.4f})")